# EuroSAT 数据集划分可视化

**目标**: 
该脚本用于读取 `split_info` 文件夹中的 `train.csv`, `valid.csv`, `test.csv` 文件，并生成一个堆叠条形图，以直观地展示每个类别在训练、验证和测试集中的分布情况。

**作用**: 
1.  **验证划分比例**：确认训练/验证/测试集的比例是否符合预期 (70%/15%/15%)。
2.  **检查分层抽样**：通过观察每个类别条形图中不同颜色的比例，可以直观地判断分层抽样是否成功，即每个集合中的类别分布是否均衡。

**操作指南**: 
1.  确保您已经成功运行了 `data_preprocessing.ipynb` 脚本。
2.  确认下方单元格中的 `metadata_path` 路径设置正确。
3.  从上到下依次运行所有单元格。

In [ ]:
# 导入必要的库
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

## 1. 配置元数据路径

此路径应指向包含 `train.csv` 等文件的 `split_info` 文件夹。

In [ ]:
# --- 关键配置区 ---

# 包含 split_info 文件夹的父目录
project_path = "F:/vscode project/IMA_PW3/EuroSat_classification-main_data_wx/"

# 包含 CSV 和 JSON 文件的元数据文件夹路径
metadata_path = os.path.join(project_path, "split_info/").replace('\\', '/')

print(f"将从以下路径读取元数据文件: {metadata_path}")

## 2. 加载数据并统计类别分布

In [ ]:
# --- 步骤 1: 加载所有 CSV 文件 ---
try:
    train_df = pd.read_csv(os.path.join(metadata_path, 'train.csv'))
    valid_df = pd.read_csv(os.path.join(metadata_path, 'valid.csv'))
    test_df = pd.read_csv(os.path.join(metadata_path, 'test.csv'))
except FileNotFoundError as e:
    print(f"错误: {e}\n请确保您已经运行了 data_preprocessing.ipynb 脚本。")
    # 如果文件不存在，则停止执行
    raise

# --- 步骤 2: 加载标签映射文件，用于将数字标签转换回类别名称 ---
with open(os.path.join(metadata_path, 'label_map.json'), 'r') as f:
    label_to_index = json.load(f)
# 创建一个反向映射，从索引到名称
index_to_label = {v: k for k, v in label_to_index.items()}

# --- 步骤 3: 统计每个数据集中各类别 Frequence ---
# value_counts() 会返回一个包含每个标签出现次数的 Series
train_counts = train_df['Label'].value_counts().sort_index()
valid_counts = valid_df['Label'].value_counts().sort_index()
test_counts = test_df['Label'].value_counts().sort_index()

# --- 步骤 4: 将统计结果合并到一个 DataFrame 中，方便绘图 ---
distribution_df = pd.DataFrame({
    'Train': train_counts,
    'Validation': valid_counts,
    'Test': test_counts
})

# 将索引从数字标签 (0, 1, 2...) 替换为类别名称 (AnnualCrop, Forest...)
distribution_df = distribution_df.rename(index=index_to_label)

# 打印 DataFrame 查看结果
print("每个数据集中各类别 Frequence:")
print(distribution_df)

## 3. 绘制堆叠条形图

In [ ]:
# 设置绘图风格
sns.set_style("whitegrid")
plt.rcParams['font.sans-serif'] = ['SimHei'] # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False # 用来正常显示负号

# --- 开始绘图 ---
ax = distribution_df.plot(
    kind='bar', 
    stacked=True, 
    figsize=(14, 8), 
    colormap='viridis', # 使用视觉效果好的颜色方案
    edgecolor='white'
)

# --- 美化图表 ---
plt.title('EuroSAT 数据集划分分布', fontsize=18, pad=20)
plt.xlabel('土地利用类别', fontsize=14, labelpad=15)
plt.ylabel('图片数量', fontsize=14, labelpad=15)
plt.xticks(rotation=45, ha="right", fontsize=12) # 旋转X轴标签，使其不重叠
plt.yticks(fontsize=12)
plt.legend(title='数据集', fontsize=12)

# --- 在每个条形块上添加具体的 Frequence ---
# 遍历图中的每个容器（每个容器对应'Train', 'Validation', 'Test'的一组条形）
for container in ax.containers:
    # 使用 bar_label 自动在条形中心添加标签
    ax.bar_label(container, label_type='center', color='white', fontsize=10, fontweight='bold')

plt.tight_layout() # 调整布局，防止标签被截断
plt.show()

### 图表解读

- **每根完整的条形** 代表一个类别的总图片数量。
- **条形中的不同颜色** 代表该类别在 `Train` (训练集), `Validation` (验证集), 和 `Test` (测试集) 中的分布情况。
- **观察**：由于我们使用了分层抽样 (`stratify`)，您应该会看到每根条形中，不同颜色的块所占的**比例**都非常相似。这直观地证明了我们的数据集划分是均衡且成功的。